# 📅 Day 04: Advanced Union-Find

---

## What Makes This "Advanced"?

Basic Union-Find (Chapter 05 Graphs) handles static connectivity — build a graph, check if two nodes are connected. **Advanced UF** tackles:

1. **Online/Dynamic connectivity** — edges added one at a time, answer queries after each
2. **Component size tracking** — "what's the largest connected group?"
3. **Creative applications** — consecutive sequences, account merging

The power of UF is that it answers connectivity queries in near-O(1) with path compression + union by rank. DFS/BFS would need to re-traverse each time.

```
╔═══════════════════════════════════════════════════════════════════════╗
║  WHY UF OVER DFS/BFS?                                              ║
║                                                                     ║
║  DFS: O(V+E) per query — fine for one-time, bad for repeated      ║
║  UF:  O(α(n)) ≈ O(1) per query — handles dynamic additions       ║
║                                                                     ║
║  If edges are added ONLINE and queries come BETWEEN additions,     ║
║  Union-Find is the tool. DFS can't do this efficiently.            ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
class UnionFind:
    """Full-featured UF with path compression, union by rank, and size tracking."""
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.size = [1] * n      # Size of each component
        self.components = n       # Number of connected components
    
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # Path compression
            x = self.parent[x]
        return x
    
    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px == py:
            return False  # Already connected
        # Union by rank
        if self.rank[px] < self.rank[py]:
            px, py = py, px
        self.parent[py] = px
        self.size[px] += self.size[py]  # Track component size
        if self.rank[px] == self.rank[py]:
            self.rank[px] += 1
        self.components -= 1
        return True

---

## 🎯 Problem 1: Number of Islands II (LC #305)

Land cells are added one at a time. After each addition, report the number of islands.

### Why UF is Perfect Here

With DFS, you'd need to re-scan the entire grid after every addition — O(m*n) per step. With UF: each addition is O(1) amortized. Add the cell, union with any adjacent land, done.

```
  Grid 3x3, positions added: (0,0), (0,1), (1,2), (2,1)
  
  After (0,0): 1 island     After (0,1): 1 island (merged)
    [1 0 0]                    [1 1 0]
    [0 0 0]                    [0 0 0]
    [0 0 0]                    [0 0 0]
  
  After (1,2): 2 islands    After (2,1): 3 islands
    [1 1 0]                    [1 1 0]
    [0 0 1]                    [0 0 1]
    [0 0 0]                    [0 1 0]
```

In [ ]:
def numIslands2(m, n, positions):
    """
    UF: add land, union with neighbors.
    Time: O(k * α(mn)) per operation, Space: O(mn)
    """
    uf = UnionFind(m * n)
    uf.components = 0  # Start with 0 islands
    grid = set()       # Track which cells are land
    result = []
    
    for r, c in positions:
        if (r, c) in grid:  # Duplicate addition
            result.append(uf.components)
            continue
        
        grid.add((r, c))
        uf.components += 1  # New island (might merge with neighbors)
        idx = r * n + c
        
        for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
            nr, nc = r + dr, c + dc
            if (nr, nc) in grid:
                uf.union(idx, nr * n + nc)  # Merge — may reduce component count
        
        result.append(uf.components)
    
    return result

print(numIslands2(3, 3, [[0,0],[0,1],[1,2],[2,1]]))  # [1,1,2,3]

---

## 🎯 Problem 2: Longest Consecutive Sequence (LC #128)

Find the length of the longest consecutive elements sequence in O(n).

### Two Approaches

**Set approach (simpler):** For each number, only start counting if `num-1` is not in the set (so you're at the START of a sequence). Then count upward.

**UF approach:** For each number, union with `num+1` if it exists. Track component sizes.

The set approach is cleaner for interviews — mentioning UF shows depth but the set version is faster to code.

In [ ]:
def longestConsecutive(nums):
    """
    Set + smart iteration. Only start from sequence beginnings.
    Time: O(n), Space: O(n)
    
    Why O(n)? Each number is visited at most twice:
    once in the outer loop, once in the inner while.
    """
    num_set = set(nums)
    longest = 0
    
    for num in num_set:
        # Only start from the BEGINNING of a sequence
        if num - 1 not in num_set:
            length = 1
            while num + length in num_set:
                length += 1
            longest = max(longest, length)
    
    return longest

print(longestConsecutive([100,4,200,1,3,2]))  # 4 → [1,2,3,4]
print(longestConsecutive([0,3,7,2,5,8,4,6,0,1]))  # 9 → [0..8]

---

## 📊 Pattern Map

```
╔═══════════════════════════════════════════════════════════════════════╗
║  ADVANCED UNION-FIND PATTERNS                                      ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  "Online connectivity"      → UF (DFS can't do this efficiently)  ║
║  "Count islands as added"   → UF + component counter              ║
║  "Merge accounts/groups"    → UF on shared keys                   ║
║  "Longest consecutive"      → Set approach (simpler than UF)      ║
║  "Equality constraints"     → UF for ==, then check !=           ║
║                                                                     ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 305 | Number of Islands II | Hard | UF + online land additions |
| 128 | Longest Consecutive Sequence | Medium | Set + start-of-sequence check |
| 721 | Accounts Merge | Medium | UF on email indices |
| 990 | Satisfiability of Equality | Medium | UF for ==, then verify != |
| 399 | Evaluate Division | Medium | Weighted UF or BFS |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 UF for dynamic/online connectivity — DFS can't do this fast  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Track component sizes in union() for "largest group" queries  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Longest Consecutive: Set approach is cleaner than UF          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Account merging: UF on shared attributes (emails, etc.)       │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Monotonic Structures